# 🧬 Birth–Death Acceleration of Diffusion Sampling: Comprehensive Multi-Regime Benchmark
**Course:** IE 643 - Deep Learning Theory and Practice  
**Author:** S Sabarinath, IIT Bombay  

---

### Overview & Objectives
This notebook provides a complete, unified experimental pipeline to evaluate all **Birth–Death (BD) acceleration algorithms** across a comprehensive sweep of step budgets ($T \in \{10, 25, 50\}$) across three distinct empirical regimes:
1. **2D Analytic Gaussian Mixture Ring**: Ground-truth 8-mode multimodal distribution in $\mathbb{R}^2$ with exact score and closed-form density.
2. **CIFAR-10 Deep Generative Diffusion**: Unconditional image synthesis in $\mathbb{R}^{3072}$ using `google/ddpm-cifar10-32`.
3. **CelebA-HQ High-Resolution Face Synthesis**: Realistic face generation in $\mathbb{R}^{196608}$ ($256 \times 256 \times 3$) using `google/ddpm-celebahq-256`.

---

### Evaluated Methods (All 6 Core Formulations):
1. **Baseline DDIM (Control)** ($\eta=0.0$ and $\eta=1.0$): Standard deterministic and stochastic DDIM sampling without birth–death interventions.
2. **Pixel-KDE Birth–Death**: Empirical target density KDE with adaptive cloning/annihilation in raw pixel space.
3. **Reduced-KDE Birth–Death**: Dimensionality-reduced target density via deep U-Net / PCA bottleneck ($d=25$) to eliminate distance concentration.
4. **Exact Flow Birth–Death**: Continuous Normalizing Flow (CNF) formulation evaluated with instantaneous change of variables and divergence tracking.
5. **SMC Reverse Transition Weighting**: $O(N)$ analytical reverse kernel likelihood without reference datasets or pairwise distance matrices.
6. **Particle Filtering Solution Space Explorer**: Stochastic branching ($N \to N \cdot B$), score drift potential, and elite resampling.

---

### Key Metrics Computed:
- **Quality**: RBF MMD$^2$ (Maximum Mean Discrepancy), Clean Target NLL, Mode Coverage, and Mode Balance TV.
- **Efficiency**: Throughput (img/s), Latency (sec/sample), and NFE (Number of Function Evaluations).
- **Diversity**: Population Uniqueness Fraction (pairwise MAE $< 0.05$) and Step-by-Step Diversity Trajectory tracking to detect the exact onset of particle repetition.
- **Automated Packaging**: All metrics (CSV, JSON), image grids, and comparative plots are saved to `all_bd_benchmark_results/` and compressed into a downloadable `.zip` file.


In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT SETUP & FAST DATA MIRRORS
# ==============================================================================
import os
import sys
import time
from pathlib import Path
import torch

print(f"[+] Python Version: {sys.version.split()[0]}")
print(f"[+] PyTorch Version: {torch.__version__}")
print(f"[+] CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[+] Device Name: {torch.cuda.get_device_name(0)}")
    print(f"[+] VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

# 1. Switch to Kaggle working directory
%cd /kaggle/working

# 2. Clone or locate repository
REPO_DIR = Path("/kaggle/working/Birth-Death-acceleration-of-diffusion-sampling")
if not REPO_DIR.exists():
    print("[+] Cloning repository from GitHub...")
    !git clone https://github.com/Aryaman-Prasad/Birth-Death-acceleration-of-diffusion-sampling.git
else:
    print("[+] Repository already present. Fetching latest updates...")
    !cd /kaggle/working/Birth-Death-acceleration-of-diffusion-sampling && git pull || true

# 3. Enter repository root (using %cd so subsequent cells inherit directory)
%cd /kaggle/working/Birth-Death-acceleration-of-diffusion-sampling

# 4. Install essential packages
!pip install -q diffusers scipy matplotlib pandas seaborn huggingface_hub

# 5. Insert src and tests into sys.path
for p in [Path.cwd() / "src", Path.cwd() / "tests", REPO_DIR / "src", REPO_DIR / "tests"]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# 6. Fast CIFAR-10 mirror (bypasses slow cs.toronto.edu 60 kB/s bottleneck -> 3-5 seconds download)
from torchvision import datasets as tv_datasets
tv_datasets.CIFAR10.url = (
    "https://github.com/Digital-Media/cv_data/releases/download/cifar-10/cifar-10-python.tar.gz"
)

# 7. Fast CelebA-HQ 256 dataset download via Kaggle internal CDN (kagglehub)
try:
    import kagglehub
    celeba_hq_path = kagglehub.dataset_download("badasstechie/celebahq-resized-256x256")
    os.environ["CELEBA_HQ_PATH"] = str(celeba_hq_path)
    print(f"[+] CelebA-HQ 256 dataset ready at: {celeba_hq_path}")
except Exception as e:
    print(f"[i] Optional kagglehub CelebA download note: {e}")

print("\n[✓] Environment configured successfully!")


In [ ]:
# ==============================================================================
# CELL 2: GLOBAL BENCHMARK CONFIGURATION (Customizable)
# ==============================================================================
# Step counts to evaluate across all regimes
STEPS_TOY = [10, 25, 50]          # Step budgets for 2D toy mixture
STEPS_CIFAR = [10, 25, 50]        # Step budgets for CIFAR-10
STEPS_CELEBA = [10, 25, 50]       # Step budgets for CelebA-HQ

# Regime selection flags (toggle to run specific parts)
RUN_TOY = True
RUN_CIFAR10 = True
RUN_CELEBA = True

# Sampling batch sizes
COUNT_TOY = 256                   # Number of particles in 2D toy experiment
BATCH_SIZE_CIFAR = 16             # Particle batch size for CIFAR-10
BATCH_SIZE_CELEBA = 16            # Particle batch size for CelebA-HQ (produces 4x4 grid of 16 faces)
REFERENCE_COUNT = 512             # Real image reference count for empirical targets & MMD

# Gentle Birth-Death Acceleration Hyperparameters (proven stable against mode collapse)
BD_TOTAL_TIME = 0.01              # Gentle pseudo-time budget tau for Birth-Death
BD_MAX_TIME = 0.005               # Maximum delta_tau per discrete BD intervention
BD_MIN_TIMESTEP = 100             # Minimum diffusion timestep for BD intervention (stops before clean image)
BD_EVERY_K = 2                    # Gentle intervention frequency (applies every 2 steps)
BD_MAX_EVENT_PROB = 0.25          # Maximum fraction of particles permitted to clone/die per step
ADD_NOISE = 0.05                  # Diversity noise scale injected into cloned particles
KDE_BANDWIDTH = 0.3               # Bandwidth for Kernel Density Estimation
BRANCH_FACTOR = 2                 # Candidate branching factor for Particle Filtering Explorer
KDE_REDUCE_DIM = 25               # Compressed latent dimension for Reduced-KDE

# Output directory structure
OUTPUT_DIR = Path("/kaggle/working/all_bd_benchmark_results")
for sub in ["toy", "cifar10", "celeba", "plots", "tables"]:
    (OUTPUT_DIR / sub).mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[+] Evaluation Device: {DEVICE}")
print(f"[+] Results Output Directory: {OUTPUT_DIR}")
print(f"[+] Step budgets: Toy={STEPS_TOY}, CIFAR-10={STEPS_CIFAR}, CelebA={STEPS_CELEBA}")
print(f"[+] Birth-Death config: total_time={BD_TOTAL_TIME}, max_time={BD_MAX_TIME}, every_k={BD_EVERY_K}, min_t={BD_MIN_TIMESTEP}, add_noise={ADD_NOISE}")


In [ ]:
# ==============================================================================
# CELL 3: METRIC UTILITIES, DIVERSITY DETECTOR, & PLOTTING HELPERS
# ==============================================================================
import json
import csv
from dataclasses import replace
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torchvision.utils import save_image, make_grid
from IPython.display import display, Image, FileLink

from diffusion.metrics import measure_sampling, rbf_mmd_squared
from diffusion.acceleration.base import SamplingState
import diffusion.pipeline as dp

def measure_particle_diversity(samples: torch.Tensor, threshold: float = 0.05) -> tuple[float, float, float]:
    # Measures uniqueness fraction, minimum pairwise MAE, and mean pairwise MAE
    N = samples.shape[0]
    if N <= 1:
        return 1.0, 1.0, 1.0

    s = samples.detach().float()
    if s.ndim == 4:
        s_min = s.amin(dim=(1, 2, 3), keepdim=True)
        s_max = s.amax(dim=(1, 2, 3), keepdim=True)
        denom = (s_max - s_min).clamp_min(1e-6)
        normed = (s - s_min) / denom
        flat = normed.reshape(N, -1)
    else:
        col_min = s.amin(dim=0, keepdim=True)
        col_max = s.amax(dim=0, keepdim=True)
        denom = (col_max - col_min).clamp_min(1e-6)
        normed = (s - col_min) / denom
        flat = normed.reshape(N, -1)

    diff = (flat.unsqueeze(1) - flat.unsqueeze(0)).abs().mean(dim=-1)
    diff.fill_diagonal_(1.0)

    visited = set()
    unique_count = 0
    for i in range(N):
        if i in visited:
            continue
        unique_count += 1
        visited.add(i)
        duplicates = (diff[i] < threshold).nonzero(as_tuple=True)[0].tolist()
        for d in duplicates:
            visited.add(d)

    off_diag_mask = ~torch.eye(N, dtype=torch.bool, device=samples.device)
    off_diag_diffs = diff[off_diag_mask]
    min_diff = float(off_diag_diffs.min().item()) if off_diag_diffs.numel() > 0 else 0.0
    mean_diff = float(off_diag_diffs.mean().item()) if off_diag_diffs.numel() > 0 else 0.0

    return float(unique_count / N), min_diff, mean_diff

def unique_fraction(samples: torch.Tensor, threshold: float = 0.05) -> float:
    # Calculates the ratio of perceptually distinct particles in a batch.
    u_frac, min_diff, mean_diff = measure_particle_diversity(samples, threshold=threshold)
    return round(float(u_frac), 4)

def compute_mmd(samples: torch.Tensor, references: torch.Tensor,
                bandwidth_factors: tuple[float, ...] = (0.5, 1.0, 2.0)) -> float:
    # Robust multi-scale RBF MMD-squared computation
    try:
        res = rbf_mmd_squared(samples, references, bandwidth_factors)
        if isinstance(res, tuple):
            metrics, _ = res
            return float(metrics["empirical_target_mmd2_rbf_multiscale"])
        elif isinstance(res, dict):
            return float(res.get("empirical_target_mmd2_rbf_multiscale", 0.0))
        elif hasattr(res, "item"):
            return float(res.item())
        return float(res)
    except Exception:
        x = samples.detach().reshape(samples.shape[0], -1).float()
        y = references.detach().reshape(references.shape[0], -1).float().to(x.device)
        d_xx = torch.cdist(x, x).square()
        d_yy = torch.cdist(y, y).square()
        d_xy = torch.cdist(x, y).square()
        pdist = torch.pdist(y, p=2)
        pos = pdist[pdist > 0]
        base_bw = max(float(pos.median().item()) if pos.numel() > 0 else 1.0, 1e-6)
        vals = []
        for factor in bandwidth_factors:
            bw = base_bw * factor
            denom = 2.0 * bw * bw
            k_xx = torch.exp(-d_xx / denom)
            k_yy = torch.exp(-d_yy / denom)
            k_xy = torch.exp(-d_xy / denom)
            vals.append(max(0.0, float((k_xx.mean() + k_yy.mean() - 2.0 * k_xy.mean()).item())))
        return float(sum(vals) / len(vals))

class BenchmarkSamplingPipeline(dp.SamplingPipeline):
    # Self-contained sampling pipeline supporting step-by-step diversity trajectory tracking
    def __init__(self, sampler, scheduler, modifiers=None, track_diversity=False, diversity_threshold=0.05):
        super().__init__(sampler, scheduler, modifiers=modifiers)
        self.track_diversity = track_diversity
        self.diversity_threshold = diversity_threshold
        self.diversity_trajectory = []

    @torch.no_grad()
    def sample(self, model, shape, device, num_inference_steps=None,
               generator=None, modifier_generator=None, track_diversity=None):
        model.eval()
        self.scheduler.to(device)
        timesteps = self.sampler.inference_timesteps(self.scheduler, num_inference_steps, device)
        samples = self.sampler.initial_samples(shape, device, self.scheduler, timesteps, generator)

        do_track = self.track_diversity if track_diversity is None else track_diversity
        if do_track:
            self.diversity_trajectory = []

        steps = []
        for index, timestep_tensor in enumerate(timesteps):
            timestep = int(timestep_tensor.item())
            previous = int(timesteps[index + 1].item()) if index + 1 < len(timesteps) else -1
            context = self.sampler.sampling_context(timestep, previous, self.scheduler)
            steps.append((timestep, previous, context))
        num_steps = len(steps)
        has_clean_endpoint = bool(steps and steps[-1][2].is_clean_sample)
        steps = [
            (timestep, previous, replace(
                context,
                step_index=index,
                num_steps=num_steps,
                has_clean_endpoint=has_clean_endpoint,
            ))
            for index, (timestep, previous, context) in enumerate(steps)
        ]

        for index, (timestep, previous, context) in enumerate(steps):
            source_samples = samples
            samples = self.sampler.step(model, samples, timestep, previous, self.scheduler, generator)

            div_before = None
            if do_track and samples.shape[0] > 1:
                div_before, _, _ = measure_particle_diversity(samples, self.diversity_threshold)

            state = SamplingState(
                samples, timestep, previous,
                self.sampler.sampling_metadata(timestep, previous, self.scheduler),
                context=context,
                source_samples=source_samples,
            )
            for modifier in self.modifiers:
                state = modifier.apply(
                    state, model=model, scheduler=self.scheduler,
                    generator=(modifier_generator if modifier_generator is not None else generator),
                )
            samples = state.samples

            if do_track and samples.shape[0] > 1:
                div_after, min_d, mean_d = measure_particle_diversity(samples, self.diversity_threshold)
                rep_started = (div_before is not None and div_before == 1.0 and div_after < 1.0)
                self.diversity_trajectory.append({
                    "step_index": index,
                    "timestep": timestep,
                    "diversity_before": round(div_before, 4) if div_before is not None else 1.0,
                    "diversity_after": round(div_after, 4),
                    "repetition_started_here": bool(rep_started),
                })

        return samples.clamp(-1, 1) if samples.ndim == 4 else samples

# Alias so all downstream cells use BenchmarkSamplingPipeline seamlessly
SamplingPipeline = BenchmarkSamplingPipeline

def save_and_show_grid(tensor_samples: torch.Tensor, save_path: Path, nrow: int = 4, title: str = None):
    # Normalized image grid saved and displayed cleanly
    clamped = (tensor_samples.detach().cpu().clamp(-1.0, 1.0) + 1.0) * 0.5
    grid = make_grid(clamped, nrow=nrow, padding=2)
    save_image(grid, save_path)
    if title:
        plt.figure(figsize=(6, 6))
        np_grid = grid.permute(1, 2, 0).numpy()
        if np_grid.shape[2] == 1:
            plt.imshow(np_grid.squeeze(-1), cmap="gray")
        else:
            plt.imshow(np_grid)
        plt.title(title, fontsize=12, fontweight="bold")
        plt.axis("off")
        plt.tight_layout()
        plt.show()

print("[✓] Metric and plotting helper functions defined.")


In [ ]:
# ==============================================================================
# CELL 4: PART 1 - TOY 2D ANALYTIC GAUSSIAN MIXTURE BENCHMARK
# ==============================================================================
if RUN_TOY:
    print("=" * 80)
    print("PART 1: 2D ANALYTIC GAUSSIAN MIXTURE RING BENCHMARK (ALL 6 METHODS)")
    print("=" * 80)

    from diffusion.acceleration import (
        BirthDeathAcceleration,
        FlowBirthDeathAcceleration,
        ParticleFilteringSolutionSpaceExplorer,
        ReducedKDEBirthDeathAcceleration,
        SMCTransitionLikelihoodAcceleration,
    )
    from diffusion.models.toy_mixture_oracle import ToyMixtureOracleModel
    from diffusion.samplers import DDIMSampler
    from diffusion.schedulers.linear import LinearScheduler
    from diffusion.targets.toy_mixture import ToyGaussianMixtureTarget

    # 1. Initialize exact analytic 8-mode mixture target and oracle model
    target_2d = ToyGaussianMixtureTarget(radius=2.0, standard_deviation=0.15, num_modes=8).to(DEVICE)
    scheduler_2d = LinearScheduler(num_train_timesteps=1000, beta_start=1e-4, beta_end=0.02).to(DEVICE)
    model_2d = ToyMixtureOracleModel(target_2d, scheduler_2d.alpha_bars).to(DEVICE)

    # 2. Ground-truth reference samples from exact target
    ref_gen = torch.Generator(device=DEVICE).manual_seed(12345)
    mode_idx = torch.randint(8, (COUNT_TOY,), device=DEVICE, generator=ref_gen)
    ref_noise = torch.randn((COUNT_TOY, 2), device=DEVICE, generator=ref_gen)
    references_2d = target_2d.centers[mode_idx] + target_2d.standard_deviation.to(target_2d.centers) * ref_noise

    # Diagnostic quality evaluator for 2D mixture
    def evaluate_toy_quality(samples: torch.Tensor, target: ToyGaussianMixtureTarget, references: torch.Tensor):
        s = samples.to(target.centers)
        clean_log_prob = target.log_prob_at_noise(s, sigma=torch.tensor(0.0, device=s.device))
        dist_sq = torch.cdist(s, target.centers).square()
        nearest_modes = dist_sq.argmin(dim=1)
        counts = torch.bincount(nearest_modes, minlength=8)
        freqs = counts.float() / s.shape[0]
        unif = 1.0 / 8.0
        mode_coverage = float((counts > 0).float().mean().item())
        mode_tv = float(0.5 * (freqs - unif).abs().sum().item())
        mmd2 = compute_mmd(s, references)
        return {
            "clean_target_nll": round(float(-clean_log_prob.mean().item()), 4),
            "mmd2": round(mmd2, 5),
            "mode_coverage": round(mode_coverage, 3),
            "mode_balance_tv": round(mode_tv, 4),
            "mode_counts": [int(c) for c in counts.cpu().tolist()],
        }

    toy_results = []
    saved_toy_samples = {}

    # Define method factories
    def build_toy_pipeline(method_name: str, steps: int):
        eta = 0.0 if method_name == "baseline_ddim" else 1.0
        sampler = DDIMSampler(eta=eta)
        modifier = None

        if method_name == "baseline_ddim":
            modifier = None
        elif method_name == "pixel_kde_bd":
            modifier = BirthDeathAcceleration(
                target_2d, kernel_bandwidth=KDE_BANDWIDTH, every_k_steps=BD_EVERY_K,
                total_time=BD_TOTAL_TIME, min_timestep=BD_MIN_TIMESTEP,
                max_delta_tau=BD_MAX_TIME, max_event_probability=BD_MAX_EVENT_PROB,
                add_noise=ADD_NOISE,
            )
        elif method_name == "reduced_kde_bd":
            modifier = ReducedKDEBirthDeathAcceleration(
                target_2d, kernel_bandwidth=KDE_BANDWIDTH, kde_reduce=2, backbone="pca",
                every_k_steps=BD_EVERY_K, total_time=BD_TOTAL_TIME, min_timestep=BD_MIN_TIMESTEP,
                max_delta_tau=BD_MAX_TIME, max_event_probability=BD_MAX_EVENT_PROB,
                add_noise=ADD_NOISE, noise_references=False,
            )
        elif method_name == "flow_bd":
            modifier = FlowBirthDeathAcceleration(
                target_2d, every_k_steps=BD_EVERY_K, total_time=BD_TOTAL_TIME,
                bandwidth=KDE_BANDWIDTH, min_timestep=BD_MIN_TIMESTEP,
                max_delta_tau=BD_MAX_TIME, max_event_probability=BD_MAX_EVENT_PROB,
                add_noise=ADD_NOISE,
            )
        elif method_name == "smc_reverse":
            modifier = SMCTransitionLikelihoodAcceleration(
                every_k_steps=BD_EVERY_K, total_time=BD_TOTAL_TIME, min_timestep=BD_MIN_TIMESTEP,
                max_delta_tau=BD_MAX_TIME, max_event_probability=BD_MAX_EVENT_PROB,
                add_noise=ADD_NOISE,
            )
        elif method_name == "particle_filter":
            modifier = ParticleFilteringSolutionSpaceExplorer(
                branch_factor=BRANCH_FACTOR, every_k_steps=BD_EVERY_K, total_time=BD_TOTAL_TIME,
                min_timestep=BD_MIN_TIMESTEP, max_delta_tau=BD_MAX_TIME,
                max_event_probability=BD_MAX_EVENT_PROB, add_noise=ADD_NOISE,
            )
        modifiers = [] if modifier is None else [modifier]
        return SamplingPipeline(sampler, scheduler_2d, modifiers=modifiers), eta

    toy_methods = [
        "baseline_ddim",
        "pixel_kde_bd",
        "reduced_kde_bd",
        "flow_bd",
        "smc_reverse",
        "particle_filter",
    ]

    for st in STEPS_TOY:
        print(f"\n---> Running Toy Benchmark for Step Budget: T = {st} <---")
        for method in toy_methods:
            pipeline, eta = build_toy_pipeline(method, st)
            gen = torch.Generator(device=DEVICE).manual_seed(42)
            mod_gen = torch.Generator(device=DEVICE).manual_seed(100042)
            shape = (COUNT_TOY, 2)

            samples, timed = measure_sampling(
                model_2d, DEVICE, COUNT_TOY, st,
                lambda: pipeline.sample(model_2d, shape, DEVICE, st, gen, mod_gen),
            )
            quality = evaluate_toy_quality(samples, target_2d, references_2d)
            u_frac = unique_fraction(samples, threshold=0.05)

            rec = {
                "regime": "toy_2d_ring",
                "method": method,
                "steps": st,
                "eta": eta,
                "throughput_samples_per_sec": round(timed.samples_per_second, 2),
                "latency_sec_per_sample": round(timed.seconds_per_sample, 5),
                "nfe": timed.nfe,
                "uniqueness": u_frac,
                **quality,
            }
            toy_results.append(rec)
            saved_toy_samples[f"{method}_{st}"] = samples.detach().cpu().numpy()
            print(f"  [{method:20s}] Steps={st:2d} | MMD2={rec['mmd2']:.5f} | NLL={rec['clean_target_nll']:.3f} | Coverage={rec['mode_coverage']:.2f} | Unique={u_frac:.2f} | Thrpt={rec['throughput_samples_per_sec']:.0f} s/s")

    # Save CSV & JSON
    toy_df = pd.DataFrame(toy_results)
    toy_df.to_csv(OUTPUT_DIR / "toy" / "toy_benchmark_results.csv", index=False)
    with open(OUTPUT_DIR / "toy" / "toy_metrics.json", "w") as f:
        json.dump(toy_results, f, indent=2)

    # Plot 2D scatter comparison at 25 steps
    target_centers = target_2d.centers.cpu().numpy()
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    step_plot = 25 if 25 in STEPS_TOY else STEPS_TOY[0]

    for i, method in enumerate(toy_methods):
        ax = axes[i]
        key = f"{method}_{step_plot}"
        if key in saved_toy_samples:
            s = saved_toy_samples[key]
            ax.scatter(s[:, 0], s[:, 1], alpha=0.6, s=16, c="royalblue", label="Samples")
            ax.scatter(target_centers[:, 0], target_centers[:, 1], c="crimson", marker="x", s=100, label="8 Modes")
            circle = plt.Circle((0, 0), 2.0, color="gray", fill=False, linestyle="--", alpha=0.5)
            ax.add_patch(circle)
            matching = [r for r in toy_results if r["method"] == method and r["steps"] == step_plot]
            if matching:
                m_rec = matching[0]
                ax.set_title(f"{method}\nMMD2={m_rec['mmd2']:.4f} | Cov={m_rec['mode_coverage']:.2f} | Uniq={m_rec['uniqueness']:.2f}", fontsize=10)
            else:
                ax.set_title(method, fontsize=10)
            ax.set_xlim(-3.5, 3.5)
            ax.set_ylim(-3.5, 3.5)
            ax.set_aspect("equal")
            ax.grid(True, linestyle=":", alpha=0.5)
            if i == 0:
                ax.legend(loc="upper right", fontsize=8)

    plt.suptitle(f"2D Gaussian Mixture Ring Particle Distributions at T={step_plot} Steps", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "plots" / "toy_ring_scatter_comparison.png", dpi=200)
    plt.show()
    print(f"\n[✓] Toy benchmark completed! Results saved to {OUTPUT_DIR / 'toy'}")


In [ ]:
# ==============================================================================
# CELL 5: PART 2 - CIFAR-10 DEEP DIFFUSION BENCHMARK (ALL 6 METHODS)
# ==============================================================================
if RUN_CIFAR10:
    print("=" * 80)
    print("PART 2: CIFAR-10 GENERATIVE DIFFUSION BENCHMARK (ALL 6 METHODS)")
    print("=" * 80)

    from diffusion.acceleration import (
        BirthDeathAcceleration,
        FlowBirthDeathAcceleration,
        ParticleFilteringSolutionSpaceExplorer,
        ReducedKDEBirthDeathAcceleration,
        SMCTransitionLikelihoodAcceleration,
    )
    from diffusion.datasets import make_data_module
    from diffusion.models import make_model
    from diffusion.samplers import DDIMSampler
    from diffusion.schedulers.linear import LinearScheduler
    from diffusion.targets import EmpiricalNoisedDataTarget

    # 1. Load Pretrained U-Net and Scheduler
    print("[+] Loading Pretrained U-Net (google/ddpm-cifar10-32)...")
    model_cifar = make_model("huggingface-unet", pretrained_model_id="google/ddpm-cifar10-32").to(DEVICE)
    scheduler_cifar = LinearScheduler(num_train_timesteps=1000, beta_start=1e-4, beta_end=0.02).to(DEVICE)

    # 2. Load Real Reference Images for Empirical Target and MMD
    print(f"[+] Loading {REFERENCE_COUNT} real CIFAR-10 reference images (fast CDN)...")
    data_module = make_data_module("cifar10", data_dir="data", image_size=32, download=True, train=True)
    real_dataset = data_module.dataset()
    from torch.utils.data import DataLoader
    loader = DataLoader(real_dataset, batch_size=REFERENCE_COUNT, shuffle=False)
    real_batch = next(iter(loader))
    real_images = (real_batch[0] if isinstance(real_batch, (list, tuple)) else real_batch).to(DEVICE)
    print(f"[+] Reference images ready: shape={real_images.shape}, min={real_images.min():.2f}, max={real_images.max():.2f}")

    # 3. Construct Empirical Target Distribution
    empirical_target = EmpiricalNoisedDataTarget(real_images[:REFERENCE_COUNT]).to(DEVICE)

    cifar_results = []
    cifar_diversity_trajectories = {}

    def build_cifar_pipeline(method_name: str, steps: int):
        eta = 0.0 if method_name == "baseline_ddim" else 1.0
        sampler = DDIMSampler(eta=eta)
        modifier = None

        if method_name in ("baseline_ddim", "baseline_ddim_eta1"):
            modifier = None
        elif method_name == "pixel_kde_bd":
            modifier = BirthDeathAcceleration(
                empirical_target, kernel_bandwidth=KDE_BANDWIDTH, every_k_steps=BD_EVERY_K,
                total_time=BD_TOTAL_TIME, min_timestep=BD_MIN_TIMESTEP,
                max_delta_tau=BD_MAX_TIME, max_event_probability=BD_MAX_EVENT_PROB,
                add_noise=ADD_NOISE,
            )
        elif method_name == "reduced_kde_bd":
            modifier = ReducedKDEBirthDeathAcceleration(
                empirical_target, kernel_bandwidth=KDE_BANDWIDTH, kde_reduce=KDE_REDUCE_DIM,
                backbone="auto", every_k_steps=BD_EVERY_K, total_time=BD_TOTAL_TIME,
                min_timestep=BD_MIN_TIMESTEP, max_delta_tau=BD_MAX_TIME,
                max_event_probability=BD_MAX_EVENT_PROB, add_noise=ADD_NOISE,
            )
        elif method_name == "flow_bd":
            modifier = FlowBirthDeathAcceleration(
                flow_target=empirical_target, every_k_steps=BD_EVERY_K, total_time=BD_TOTAL_TIME,
                bandwidth=KDE_BANDWIDTH, min_timestep=BD_MIN_TIMESTEP,
                max_delta_tau=BD_MAX_TIME, max_event_probability=BD_MAX_EVENT_PROB,
                add_noise=ADD_NOISE,
            )
        elif method_name == "smc_reverse":
            modifier = SMCTransitionLikelihoodAcceleration(
                every_k_steps=BD_EVERY_K, total_time=BD_TOTAL_TIME, min_timestep=BD_MIN_TIMESTEP,
                max_delta_tau=BD_MAX_TIME, max_event_probability=BD_MAX_EVENT_PROB,
                add_noise=ADD_NOISE,
            )
        elif method_name == "particle_filter":
            modifier = ParticleFilteringSolutionSpaceExplorer(
                branch_factor=BRANCH_FACTOR, every_k_steps=BD_EVERY_K, total_time=BD_TOTAL_TIME,
                min_timestep=BD_MIN_TIMESTEP, max_delta_tau=BD_MAX_TIME,
                max_event_probability=BD_MAX_EVENT_PROB, add_noise=ADD_NOISE,
            )
        modifiers = [] if modifier is None else [modifier]
        pipeline = SamplingPipeline(sampler, scheduler_cifar, modifiers=modifiers)
        return pipeline, eta

    cifar_methods = [
        "baseline_ddim",
        "baseline_ddim_eta1",
        "pixel_kde_bd",
        "reduced_kde_bd",
        "flow_bd",
        "smc_reverse",
        "particle_filter",
    ]

    for st in STEPS_CIFAR:
        print(f"\n---> Running CIFAR-10 Sweep for Step Budget: T = {st} <---")
        for method in cifar_methods:
            pipeline, eta = build_cifar_pipeline(method, st)
            gen = torch.Generator(device=DEVICE).manual_seed(42)
            mod_gen = torch.Generator(device=DEVICE).manual_seed(100042)
            shape = (BATCH_SIZE_CIFAR, 3, 32, 32)

            samples, timed = measure_sampling(
                model_cifar, DEVICE, BATCH_SIZE_CIFAR, st,
                lambda: pipeline.sample(model_cifar, shape, DEVICE, st, gen, mod_gen),
            )

            # Evaluate metrics against real dataset
            u_frac = unique_fraction(samples, threshold=0.05)
            flat_gen = samples.reshape(samples.shape[0], -1)
            flat_real = real_images.reshape(real_images.shape[0], -1)
            mmd2 = compute_mmd(flat_gen, flat_real)

            rec = {
                "regime": "cifar10",
                "method": method,
                "steps": st,
                "eta": eta,
                "throughput_img_per_sec": round(timed.samples_per_second, 2),
                "latency_sec_per_sample": round(timed.seconds_per_sample, 4),
                "nfe": timed.nfe,
                "uniqueness": u_frac,
                "mmd2": round(mmd2, 5),
            }
            cifar_results.append(rec)
            cifar_diversity_trajectories[f"{method}_{st}"] = pipeline.diversity_trajectory

            # Save sample image grid
            img_path = OUTPUT_DIR / "cifar10" / f"cifar10_{method}_steps{st}.png"
            save_and_show_grid(samples, img_path, nrow=4, title=f"CIFAR-10: {method} (T={st})")
            print(f"  [{method:20s}] Steps={st:2d} | MMD2={mmd2:.5f} | Unique={u_frac:.2f} | Thrpt={rec['throughput_img_per_sec']:.1f} img/s")

    # Save CSV, JSON & Trajectories
    cifar_df = pd.DataFrame(cifar_results)
    cifar_df.to_csv(OUTPUT_DIR / "cifar10" / "cifar10_benchmark_results.csv", index=False)
    with open(OUTPUT_DIR / "cifar10" / "cifar10_metrics.json", "w") as f:
        json.dump(cifar_results, f, indent=2)
    with open(OUTPUT_DIR / "cifar10" / "diversity_trajectories.json", "w") as f:
        json.dump(cifar_diversity_trajectories, f, indent=2)

    print(f"\n[✓] CIFAR-10 benchmark completed! Results saved to {OUTPUT_DIR / 'cifar10'}")


In [ ]:
# ==============================================================================
# CELL 6: PART 2B - DEDICATED DIRECT PIXEL-PCA & LATENT PCA BENCHMARK (5, 10, 50 STEPS)
# ==============================================================================
print("=" * 80)
print("PART 2B: DEDICATED DIRECT PIXEL-PCA & LATENT PCA BENCHMARK (STEPS: 5, 10, 50)")
print("=" * 80)

import os, json, shutil, zipfile, time
from pathlib import Path

import torch
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from torchvision.utils import make_grid
from IPython.display import FileLink, display

from diffusion.acceleration import (
    BirthDeathAcceleration,
    ReducedKDEBirthDeathAcceleration,
)
from diffusion.samplers import DDIMSampler
from diffusion.pipeline import SamplingPipeline

# 1. Output directory & configuration
PCA_OUT_DIR = Path("/kaggle/working/cifar10_pca_sweep_results")
(PCA_OUT_DIR / "samples").mkdir(parents=True, exist_ok=True)
(PCA_OUT_DIR / "plots").mkdir(parents=True, exist_ok=True)
(PCA_OUT_DIR / "tables").mkdir(parents=True, exist_ok=True)

pca_eval_steps = [5, 10, 50]       # Steps to evaluate: 5, 10, 50
pca_dims = [10, 25, 50]            # Dimensions for Direct Pixel-PCA ablation
batch_size = BATCH_SIZE_CIFAR if "BATCH_SIZE_CIFAR" in globals() else 16
dev = DEVICE if "DEVICE" in globals() else torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Gentle Birth-Death settings (proven stable against mode collapse)
bd_k = BD_EVERY_K if "BD_EVERY_K" in globals() else 2
bd_tau = BD_TOTAL_TIME if "BD_TOTAL_TIME" in globals() else 0.01
bd_min_t = BD_MIN_TIMESTEP if "BD_MIN_TIMESTEP" in globals() else 100
bd_max_t = BD_MAX_TIME if "BD_MAX_TIME" in globals() else 0.005
bd_max_prob = BD_MAX_EVENT_PROB if "BD_MAX_EVENT_PROB" in globals() else 0.25
bd_noise = ADD_NOISE if "ADD_NOISE" in globals() else 0.05
bd_bw = KDE_BANDWIDTH if "KDE_BANDWIDTH" in globals() else 0.3

# Helper functions (self-contained fallbacks)
def _u_frac(samples: torch.Tensor, threshold: float = 0.05) -> float:
    b = samples.shape[0]
    if b <= 1:
        return 1.0
    flat = samples.detach().reshape(b, -1)
    diffs = (flat.unsqueeze(0) - flat.unsqueeze(1)).abs().mean(dim=-1)
    eye = torch.eye(b, dtype=torch.bool, device=samples.device)
    diffs = diffs.masked_fill(eye, float("inf"))
    duplicates = (diffs < threshold).any(dim=1)
    return float((~duplicates).sum().item()) / b

def _mmd(s1: torch.Tensor, s2: torch.Tensor) -> float:
    try:
        from diffusion.metrics import rbf_mmd_squared
        res = rbf_mmd_squared(s1, s2)
        if isinstance(res, tuple):
            return float(res[0].get("empirical_target_mmd2_rbf_multiscale", 0.0))
        return float(res)
    except Exception:
        gamma = 1.0 / (s1.shape[1] * 0.5)
        k11 = torch.exp(-gamma * torch.cdist(s1, s1).square()).mean()
        k22 = torch.exp(-gamma * torch.cdist(s2, s2).square()).mean()
        k12 = torch.exp(-gamma * torch.cdist(s1, s2).square()).mean()
        return float((k11 + k22 - 2 * k12).item())

def _save_grid(tensor_samples: torch.Tensor, save_path: Path, nrow: int = 4, title: str = None):
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with torch.no_grad():
        clamped = (tensor_samples.detach().cpu().clamp(-1.0, 1.0) + 1.0) / 2.0
        grid = make_grid(clamped, nrow=nrow, padding=2)
        grid_np = grid.permute(1, 2, 0).numpy()
    plt.figure(figsize=(7, 7))
    plt.imshow(grid_np)
    plt.axis("off")
    if title:
        plt.title(title, fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close()

def _timed_sample(model, shape, steps, sample_call):
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    out = sample_call()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = max(time.perf_counter() - t0, 1e-6)
    return out, elapsed / shape[0], shape[0] / elapsed

pca_results = []
shape = (batch_size, 3, 32, 32)
flat_real = real_images[:batch_size].reshape(batch_size, -1) if "real_images" in globals() else None

# 2. Run Direct Pixel-PCA Sweep across steps & dimensions
print(f"\n[+] Running Direct Pixel-PCA Sweep across T={pca_eval_steps} steps (Dims: {pca_dims})...")
for eval_step in pca_eval_steps:
    print(f"\n---> Direct Pixel-PCA Sweep for Step Budget: T = {eval_step} <---")
    for d in pca_dims:
        pca_mod = ReducedKDEBirthDeathAcceleration(
            empirical_target,
            kernel_bandwidth=bd_bw,
            kde_reduce=d,
            backbone="pca",
            projection_method="pca",
            every_k_steps=bd_k,
            total_time=bd_tau,
            min_timestep=bd_min_t,
            max_delta_tau=bd_max_t,
            max_event_probability=bd_max_prob,
            add_noise=bd_noise,
        )
        sampler = DDIMSampler(eta=1.0)
        pipeline = SamplingPipeline(sampler, scheduler_cifar, modifiers=[pca_mod])
        gen = torch.Generator(device=dev).manual_seed(42)
        mod_gen = torch.Generator(device=dev).manual_seed(100042)

        samples, sec_per_sample, samples_per_sec = _timed_sample(
            model_cifar, shape, eval_step,
            lambda: pipeline.sample(model_cifar, shape, dev, eval_step, gen, mod_gen)
        )

        u_frac = _u_frac(samples, threshold=0.05)
        flat_gen = samples.reshape(samples.shape[0], -1)
        mmd2 = _mmd(flat_gen, flat_real) if flat_real is not None else 0.0

        rec = {
            "regime": "cifar10_pca_study",
            "method": f"Direct Pixel-PCA (d={d})",
            "pca_type": "pixel_pca",
            "pca_dim": d,
            "steps": eval_step,
            "throughput_img_per_sec": round(samples_per_sec, 2),
            "latency_sec_per_sample": round(sec_per_sample, 4),
            "nfe": eval_step,
            "uniqueness": u_frac,
            "mmd2": round(mmd2, 5),
        }
        pca_results.append(rec)
        img_path = PCA_OUT_DIR / "samples" / f"cifar10_pixel_pca_d{d}_steps{eval_step}.png"
        _save_grid(samples, img_path, nrow=4, title=f"Direct Pixel-PCA (d={d}, T={eval_step})")
        print(f"  [Pixel-PCA d={d:2d}, T={eval_step:2d}] MMD2={mmd2:.5f} | Unique={u_frac:.2f} | Thrpt={rec['throughput_img_per_sec']:.1f} img/s")

# 3. Run U-Net Bottleneck + PCA across steps (Deep latent projection to d=25)
print(f"\n[+] Running Deep U-Net Bottleneck + PCA (d=25) across steps {pca_eval_steps}...")
for eval_step in pca_eval_steps:
    unet_pca_mod = ReducedKDEBirthDeathAcceleration(
        empirical_target,
        kernel_bandwidth=bd_bw,
        kde_reduce=25,
        backbone="bottleneck",
        projection_method="pca",
        every_k_steps=bd_k,
        total_time=bd_tau,
        min_timestep=bd_min_t,
        max_delta_tau=bd_max_t,
        max_event_probability=bd_max_prob,
        add_noise=bd_noise,
    )
    sampler = DDIMSampler(eta=1.0)
    pipeline = SamplingPipeline(sampler, scheduler_cifar, modifiers=[unet_pca_mod])
    gen = torch.Generator(device=dev).manual_seed(42)
    mod_gen = torch.Generator(device=dev).manual_seed(100042)

    samples, sec_per_sample, samples_per_sec = _timed_sample(
        model_cifar, shape, eval_step,
        lambda: pipeline.sample(model_cifar, shape, dev, eval_step, gen, mod_gen)
    )
    u_frac = _u_frac(samples, threshold=0.05)
    flat_gen = samples.reshape(samples.shape[0], -1)
    mmd2 = _mmd(flat_gen, flat_real) if flat_real is not None else 0.0

    rec_unet = {
        "regime": "cifar10_pca_study",
        "method": "U-Net Bottleneck + PCA (d=25)",
        "pca_type": "unet_pca",
        "pca_dim": 25,
        "steps": eval_step,
        "throughput_img_per_sec": round(samples_per_sec, 2),
        "latency_sec_per_sample": round(sec_per_sample, 4),
        "nfe": eval_step,
        "uniqueness": u_frac,
        "mmd2": round(mmd2, 5),
    }
    pca_results.append(rec_unet)
    img_path = PCA_OUT_DIR / "samples" / f"cifar10_unet_pca_d25_steps{eval_step}.png"
    _save_grid(samples, img_path, nrow=4, title=f"U-Net Bottleneck + PCA (d=25, T={eval_step})")
    print(f"  [U-Net Bottleneck PCA d=25, T={eval_step:2d}] MMD2={mmd2:.5f} | Unique={u_frac:.2f} | Thrpt={rec_unet['throughput_img_per_sec']:.1f} img/s")

# 4. Save PCA Study Results & Display Comparison Table
pca_df = pd.DataFrame(pca_results)
pca_df.to_csv(PCA_OUT_DIR / "tables" / "pca_benchmark_results.csv", index=False)
with open(PCA_OUT_DIR / "tables" / "pca_metrics.json", "w") as f:
    json.dump(pca_results, f, indent=2)

print("\n" + "=" * 80)
print("PCA BENCHMARK COMPARISON TABLE (Direct Pixel-PCA vs Latent U-Net PCA across steps):")
print("=" * 80)
display(pca_df)

# 5. Plot PCA Dimension Ablation Curves across Steps
plt.figure(figsize=(9, 5))
p_df = pca_df[pca_df["pca_type"] == "pixel_pca"]
u_df = pca_df[pca_df["pca_type"] == "unet_pca"]

palette = sns.color_palette("tab10", n_colors=len(pca_eval_steps))
for idx, st in enumerate(pca_eval_steps):
    sub_p = p_df[p_df["steps"] == st]
    sub_u = u_df[u_df["steps"] == st]
    if not sub_p.empty:
        plt.plot(sub_p["pca_dim"], sub_p["mmd2"], marker="o", linewidth=2.0, color=palette[idx], label=f"Pixel-PCA (T={st})")
    if not sub_u.empty:
        plt.axhline(y=sub_u["mmd2"].values[0], color=palette[idx], linestyle="--", alpha=0.7, label=f"U-Net PCA d=25 (T={st})")

plt.title("PCA Latent Dimension Ablation vs Generation Quality Across Step Budgets", fontweight="bold")
plt.xlabel("PCA Compressed Dimension (d)")
plt.ylabel("RBF MMD² (Lower is Better)")
plt.xticks(pca_dims)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig(PCA_OUT_DIR / "plots" / "pca_dimension_ablation.png", dpi=200)
plt.show()

# 6. Package all outputs into a standalone ZIP archive and display 1-click download link
zip_output_path = Path("/kaggle/working/cifar10_pca_sweep_results.zip")
if zip_output_path.exists():
    zip_output_path.unlink()

print(f"\n[+] Packaging results from {PCA_OUT_DIR} into {zip_output_path.name}...")
shutil.make_archive(
    "/kaggle/working/cifar10_pca_sweep_results",
    "zip",
    root_dir=str(PCA_OUT_DIR.parent),
    base_dir=str(PCA_OUT_DIR.name),
)

size_mb = zip_output_path.stat().st_size / (1024 * 1024)
print(f"[✓] SUCCESS: Archive created successfully ({size_mb:.2f} MB)!")
print("\n" + "=" * 80)
print("CLICK THE LINK BELOW TO DOWNLOAD PCA SWEEP RESULTS (.ZIP):")
print("=" * 80)
display(FileLink("cifar10_pca_sweep_results.zip"))


In [ ]:
# ==============================================================================
# CELL 7: PART 3 - CELEBA-HQ HIGH-RESOLUTION FACE SYNTHESIS BENCHMARK
# ==============================================================================
if RUN_CELEBA:
    print("=" * 80)
    print("PART 3: CELEBA-HQ HIGH-RESOLUTION FACE SYNTHESIS (256x256)")
    print("=" * 80)

    from diffusion.acceleration import (
        BirthDeathAcceleration,
        ParticleFilteringSolutionSpaceExplorer,
        ReducedKDEBirthDeathAcceleration,
        SMCTransitionLikelihoodAcceleration,
    )
    from diffusion.models import make_model
    from diffusion.samplers import DDIMSampler
    from diffusion.schedulers.linear import LinearScheduler
    from diffusion.targets import EmpiricalNoisedDataTarget

    # 1. Load Pretrained CelebA-HQ U-Net (256x256x3)
    print("[+] Loading Pretrained U-Net (google/ddpm-celebahq-256)...")
    model_celeba = make_model("huggingface-unet", pretrained_model_id="google/ddpm-celebahq-256").to(DEVICE)
    scheduler_celeba = LinearScheduler(num_train_timesteps=1000, beta_start=1e-4, beta_end=0.02).to(DEVICE)

    # 2. Check for optional mounted CelebA dataset on Kaggle, or create stable prior anchor
    celeba_env = os.environ.get("CELEBA_HQ_PATH", "")
    celeba_input_dirs = [
        Path(celeba_env) if celeba_env else Path("/nonexistent"),
        Path("/kaggle/input/celebahq-resized-256x256"),
        Path("/kaggle/input/celeba-dataset/img_align_celeba"),
    ]
    celeba_local_dir = next((p for p in celeba_input_dirs if p.exists()), None)
    if celeba_local_dir is not None:
        print(f"[+] Using CelebA dataset at: {celeba_local_dir}")
        from torchvision import transforms
        from torchvision.datasets import ImageFolder
        t = transforms.Compose([transforms.Resize((256, 256)), transforms.CenterCrop((256, 256)), transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
        # If directory has subfolders (ImageFolder requirement), use it directly; otherwise use parent
        try:
            ds = ImageFolder(str(celeba_local_dir), transform=t)
        except Exception:
            ds = ImageFolder(str(celeba_local_dir.parent), transform=t)
        from torch.utils.data import DataLoader
        c_loader = DataLoader(ds, batch_size=64, shuffle=True)
        dummy_refs = next(iter(c_loader))[0].to(DEVICE)
    else:
        print("[+] Dataset folder not mounted; using standard normal prior anchor.")
        dummy_refs = torch.randn((64, 3, 256, 256), device=DEVICE).clamp(-1.0, 1.0)

    anchor_target = EmpiricalNoisedDataTarget(dummy_refs).to(DEVICE)

    celeba_methods = [
        "baseline_ddim",
        "baseline_ddim_eta1",
        "smc_reverse",
        "particle_filter",
        "reduced_kde_bd",
    ]

    celeba_results = []

    def build_celeba_pipeline(method_name: str, steps: int):
        eta = 0.0 if method_name == "baseline_ddim" else 1.0
        sampler = DDIMSampler(eta=eta)
        modifier = None

        if method_name in ("baseline_ddim", "baseline_ddim_eta1"):
            modifier = None
        elif method_name == "smc_reverse":
            modifier = SMCTransitionLikelihoodAcceleration(
                every_k_steps=BD_EVERY_K, total_time=BD_TOTAL_TIME, min_timestep=BD_MIN_TIMESTEP,
                max_delta_tau=BD_MAX_TIME, max_event_probability=BD_MAX_EVENT_PROB,
                add_noise=ADD_NOISE,
            )
        elif method_name == "particle_filter":
            modifier = ParticleFilteringSolutionSpaceExplorer(
                branch_factor=BRANCH_FACTOR, every_k_steps=BD_EVERY_K, total_time=BD_TOTAL_TIME,
                min_timestep=BD_MIN_TIMESTEP, max_delta_tau=BD_MAX_TIME,
                max_event_probability=BD_MAX_EVENT_PROB, add_noise=ADD_NOISE,
            )
        elif method_name == "reduced_kde_bd":
            modifier = ReducedKDEBirthDeathAcceleration(
                anchor_target, kernel_bandwidth=KDE_BANDWIDTH, kde_reduce=KDE_REDUCE_DIM,
                backbone="pca", every_k_steps=BD_EVERY_K, total_time=BD_TOTAL_TIME,
                min_timestep=BD_MIN_TIMESTEP, max_delta_tau=BD_MAX_TIME,
                max_event_probability=BD_MAX_EVENT_PROB, add_noise=ADD_NOISE,
                noise_references=False,
            )

        modifiers = [] if modifier is None else [modifier]
        pipeline = SamplingPipeline(sampler, scheduler_celeba, modifiers=modifiers)
        return pipeline, eta

    for st in STEPS_CELEBA:
        print(f"\n---> Running CelebA-HQ Sweep for Step Budget: T = {st} <---")
        for method in celeba_methods:
            pipeline, eta = build_celeba_pipeline(method, st)
            gen = torch.Generator(device=DEVICE).manual_seed(42)
            mod_gen = torch.Generator(device=DEVICE).manual_seed(100042)
            shape = (BATCH_SIZE_CELEBA, 3, 256, 256)

            samples, timed = measure_sampling(
                model_celeba, DEVICE, BATCH_SIZE_CELEBA, st,
                lambda: pipeline.sample(model_celeba, shape, DEVICE, st, gen, mod_gen),
            )

            u_frac = unique_fraction(samples, threshold=0.05)

            rec = {
                "regime": "celeba_hq_256",
                "method": method,
                "steps": st,
                "eta": eta,
                "throughput_img_per_sec": round(timed.samples_per_second, 2),
                "latency_sec_per_sample": round(timed.seconds_per_sample, 4),
                "nfe": timed.nfe,
                "uniqueness": u_frac,
            }
            celeba_results.append(rec)

            # Save sample image grid (4x4 grid for batch size 16)
            img_path = OUTPUT_DIR / "celeba" / f"celeba_{method}_steps{st}.png"
            save_and_show_grid(samples, img_path, nrow=4, title=f"CelebA-HQ (256x256): {method} (T={st})")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            print(f"  [{method:20s}] Steps={st:2d} | Unique={u_frac:.2f} | Thrpt={rec['throughput_img_per_sec']:.2f} img/s | Latency={rec['latency_sec_per_sample']:.3f} s/img")

    # Save CSV & JSON
    celeba_df = pd.DataFrame(celeba_results)
    celeba_df.to_csv(OUTPUT_DIR / "celeba" / "celeba_benchmark_results.csv", index=False)
    with open(OUTPUT_DIR / "celeba" / "celeba_metrics.json", "w") as f:
        json.dump(celeba_results, f, indent=2)

    print(f"\n[✓] CelebA-HQ benchmark completed! Results saved to {OUTPUT_DIR / 'celeba'}")


In [ ]:
# ==============================================================================
# CELL 8: PART 3B - CELEBA-HQ DEDICATED DIRECT PIXEL-PCA & LATENT PCA STUDY
# ==============================================================================
if RUN_CELEBA:
    print("=" * 80)
    print("PART 3B: CELEBA-HQ DIRECT PIXEL-PCA (196608D -> d) & LATENT PCA BENCHMARK")
    print("=" * 80)

    from diffusion.acceleration import ReducedKDEBirthDeathAcceleration
    from diffusion.samplers import DDIMSampler

    celeba_pca_results = []
    pca_dims = [10, 25, 50]
    eval_step = 25  # Standard benchmark budget

    # 1. Direct Pixel-PCA Dimension Sweep on High-Res Faces (196608 dimensions via SVD)
    print(f"\n[+] Running CelebA-HQ Direct Pixel-PCA Dimension Ablation at T={eval_step} steps (Dims: {pca_dims})...")
    for d in pca_dims:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        pca_mod = ReducedKDEBirthDeathAcceleration(
            anchor_target,
            kernel_bandwidth=KDE_BANDWIDTH,
            kde_reduce=d,
            backbone="pca",  # Direct linear Pixel-PCA on 256x256x3 (196,608D) via SVD
            projection_method="pca",
            every_k_steps=BD_EVERY_K,
            total_time=BD_TOTAL_TIME,
            min_timestep=BD_MIN_TIMESTEP,
            max_delta_tau=BD_MAX_TIME,
            max_event_probability=BD_MAX_EVENT_PROB,
            add_noise=ADD_NOISE,
            noise_references=False,
        )
        sampler = DDIMSampler(eta=1.0)
        pipeline = SamplingPipeline(sampler, scheduler_celeba, modifiers=[pca_mod])
        gen = torch.Generator(device=DEVICE).manual_seed(42)
        mod_gen = torch.Generator(device=DEVICE).manual_seed(100042)
        shape = (BATCH_SIZE_CELEBA, 3, 256, 256)

        samples, timed = measure_sampling(
            model_celeba, DEVICE, BATCH_SIZE_CELEBA, eval_step,
            lambda: pipeline.sample(model_celeba, shape, DEVICE, eval_step, gen, mod_gen),
        )

        u_frac = unique_fraction(samples, threshold=0.05)
        flat_gen = samples.reshape(samples.shape[0], -1)
        flat_real = dummy_refs[:BATCH_SIZE_CELEBA].reshape(BATCH_SIZE_CELEBA, -1)
        mmd2 = compute_mmd(flat_gen, flat_real)

        rec = {
            "regime": "celeba_pca_study",
            "method": f"CelebA Direct Pixel-PCA (d={d})",
            "pca_type": "pixel_pca",
            "pca_dim": d,
            "steps": eval_step,
            "throughput_img_per_sec": round(timed.samples_per_second, 2),
            "latency_sec_per_sample": round(timed.seconds_per_sample, 4),
            "nfe": timed.nfe,
            "uniqueness": u_frac,
            "mmd2": round(mmd2, 5),
        }
        celeba_pca_results.append(rec)

        # Save 4x4 image grid (16 samples)
        img_path = OUTPUT_DIR / "celeba" / f"celeba_pixel_pca_d{d}_steps{eval_step}.png"
        save_and_show_grid(samples, img_path, nrow=4, title=f"CelebA-HQ Direct Pixel-PCA (d={d}, T={eval_step})")
        print(f"  [CelebA Pixel-PCA d={d:2d}] MMD2={mmd2:.5f} | Unique={u_frac:.2f} | Thrpt={rec['throughput_img_per_sec']:.2f} img/s | Latency={rec['latency_sec_per_sample']:.3f} s/img")

    # 2. U-Net Bottleneck + PCA on CelebA-HQ (Deep latent projection to d=25)
    print(f"\n[+] Running CelebA-HQ Deep U-Net Bottleneck + PCA (d=25)...")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    unet_pca_mod = ReducedKDEBirthDeathAcceleration(
        anchor_target,
        kernel_bandwidth=KDE_BANDWIDTH,
        kde_reduce=25,
        backbone="bottleneck",
        projection_method="pca",
        every_k_steps=BD_EVERY_K,
        total_time=BD_TOTAL_TIME,
        min_timestep=BD_MIN_TIMESTEP,
        max_delta_tau=BD_MAX_TIME,
        max_event_probability=BD_MAX_EVENT_PROB,
        add_noise=ADD_NOISE,
        noise_references=False,
    )
    sampler = DDIMSampler(eta=1.0)
    pipeline = SamplingPipeline(sampler, scheduler_celeba, modifiers=[unet_pca_mod])
    gen = torch.Generator(device=DEVICE).manual_seed(42)
    mod_gen = torch.Generator(device=DEVICE).manual_seed(100042)

    samples, timed = measure_sampling(
        model_celeba, DEVICE, BATCH_SIZE_CELEBA, eval_step,
        lambda: pipeline.sample(model_celeba, shape, DEVICE, eval_step, gen, mod_gen),
    )
    u_frac = unique_fraction(samples, threshold=0.05)
    flat_gen = samples.reshape(samples.shape[0], -1)
    mmd2 = compute_mmd(flat_gen, flat_real)

    rec_unet = {
        "regime": "celeba_pca_study",
        "method": "CelebA U-Net Bottleneck + PCA (d=25)",
        "pca_type": "unet_pca",
        "pca_dim": 25,
        "steps": eval_step,
        "throughput_img_per_sec": round(timed.samples_per_second, 2),
        "latency_sec_per_sample": round(timed.seconds_per_sample, 4),
        "nfe": timed.nfe,
        "uniqueness": u_frac,
        "mmd2": round(mmd2, 5),
    }
    celeba_pca_results.append(rec_unet)

    # Save 4x4 image grid (16 samples)
    img_path = OUTPUT_DIR / "celeba" / f"celeba_unet_pca_d25_steps{eval_step}.png"
    save_and_show_grid(samples, img_path, nrow=4, title=f"CelebA-HQ U-Net Bottleneck + PCA (d=25, T={eval_step})")
    print(f"  [CelebA U-Net PCA d=25] MMD2={mmd2:.5f} | Unique={u_frac:.2f} | Thrpt={rec_unet['throughput_img_per_sec']:.2f} img/s | Latency={rec_unet['latency_sec_per_sample']:.3f} s/img")

    # 3. Save CelebA PCA Study Results & Display Comparison Table
    celeba_pca_df = pd.DataFrame(celeba_pca_results)
    celeba_pca_df.to_csv(OUTPUT_DIR / "tables" / "celeba_pca_benchmark_results.csv", index=False)
    with open(OUTPUT_DIR / "tables" / "celeba_pca_metrics.json", "w") as f:
        json.dump(celeba_pca_results, f, indent=2)

    print("\n" + "=" * 80)
    print("CELEBA-HQ PCA BENCHMARK COMPARISON TABLE (Direct Pixel-PCA vs Latent U-Net PCA):")
    print("=" * 80)
    display(celeba_pca_df)

    # 4. Plot CelebA PCA Dimension Ablation Curve
    plt.figure(figsize=(8, 4.5))
    p_df = celeba_pca_df[celeba_pca_df["pca_type"] == "pixel_pca"]
    plt.plot(p_df["pca_dim"], p_df["mmd2"], marker="o", linewidth=2.5, color="royalblue", label="CelebA Direct Pixel-PCA MMD²")
    plt.axhline(y=rec_unet["mmd2"], color="crimson", linestyle="--", label="CelebA U-Net Bottleneck PCA (d=25)")
    plt.title("CelebA-HQ PCA Dimension Ablation vs Generation Quality", fontweight="bold")
    plt.xlabel("PCA Compressed Dimension (d)")
    plt.ylabel("RBF MMD² (Lower is Better)")
    plt.xticks(pca_dims)
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "plots" / "celeba_pca_dimension_ablation.png", dpi=200)
    plt.show()
    print(f"[✓] CelebA-HQ PCA benchmark completed! Results saved to {OUTPUT_DIR / 'tables' / 'celeba_pca_benchmark_results.csv'}")


In [ ]:
# ==============================================================================
# CELL 9: PART 4 - COMPARATIVE VISUALIZATIONS & SUMMARY TABLES
# ==============================================================================
print("=" * 80)
print("PART 4: COMPILING CROSS-METHOD COMPARATIVE PLOTS & TABLES")
print("=" * 80)

import seaborn as sns
sns.set_theme(style="whitegrid", font_scale=1.1)

# 1. Master DataFrame compilation
all_dfs = []
if (OUTPUT_DIR / "toy" / "toy_benchmark_results.csv").exists():
    all_dfs.append(pd.read_csv(OUTPUT_DIR / "toy" / "toy_benchmark_results.csv"))
if (OUTPUT_DIR / "cifar10" / "cifar10_benchmark_results.csv").exists():
    all_dfs.append(pd.read_csv(OUTPUT_DIR / "cifar10" / "cifar10_benchmark_results.csv"))
if (OUTPUT_DIR / "tables" / "pca_benchmark_results.csv").exists():
    all_dfs.append(pd.read_csv(OUTPUT_DIR / "tables" / "pca_benchmark_results.csv"))
if (OUTPUT_DIR / "celeba" / "celeba_benchmark_results.csv").exists():
    all_dfs.append(pd.read_csv(OUTPUT_DIR / "celeba" / "celeba_benchmark_results.csv"))
if (OUTPUT_DIR / "tables" / "celeba_pca_benchmark_results.csv").exists():
    all_dfs.append(pd.read_csv(OUTPUT_DIR / "tables" / "celeba_pca_benchmark_results.csv"))

if all_dfs:
    master_df = pd.concat(all_dfs, ignore_index=True)
    master_df.to_csv(OUTPUT_DIR / "tables" / "master_benchmark_summary.csv", index=False)
    print(f"[+] Master summary table created with {len(master_df)} experiment records.")

# 2. Publication Figure 1: Quality vs Sampler Steps (CIFAR-10 MMD^2)
if (OUTPUT_DIR / "cifar10" / "cifar10_benchmark_results.csv").exists():
    c_df = pd.read_csv(OUTPUT_DIR / "cifar10" / "cifar10_benchmark_results.csv")
    plt.figure(figsize=(9, 5))
    palette = sns.color_palette("tab10", n_colors=c_df["method"].nunique())
    sns.lineplot(data=c_df, x="steps", y="mmd2", hue="method", marker="o", linewidth=2.5, palette=palette)
    plt.title("CIFAR-10 Generation Quality: RBF MMD² vs Sampler Step Budget", fontweight="bold")
    plt.xlabel("Sampling Steps (T)")
    plt.ylabel("RBF MMD² (Lower is Better)")
    plt.xticks(sorted(c_df["steps"].unique()))
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=True)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "plots" / "cifar10_mmd_vs_steps.png", dpi=200)
    plt.show()

# 3. Publication Figure 2: Population Diversity vs Sampler Steps
if (OUTPUT_DIR / "cifar10" / "cifar10_benchmark_results.csv").exists():
    plt.figure(figsize=(9, 5))
    sns.lineplot(data=c_df, x="steps", y="uniqueness", hue="method", marker="s", linewidth=2.5, palette=palette)
    plt.title("Particle Population Diversity: Uniqueness Fraction vs Steps", fontweight="bold")
    plt.xlabel("Sampling Steps (T)")
    plt.ylabel("Unique Fraction (1.0 = No Collapse)")
    plt.ylim(0.0, 1.05)
    plt.xticks(sorted(c_df["steps"].unique()))
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=True)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "plots" / "cifar10_diversity_vs_steps.png", dpi=200)
    plt.show()

# 4. Publication Figure 3: Speed vs Quality Tradeoff (Throughput vs MMD^2)
if (OUTPUT_DIR / "cifar10" / "cifar10_benchmark_results.csv").exists():
    plt.figure(figsize=(9, 5))
    sns.scatterplot(
        data=c_df, x="throughput_img_per_sec", y="mmd2", hue="method",
        style="steps", s=180, palette=palette, alpha=0.9
    )
    plt.title("Speed vs. Quality Pareto Tradeoff (CIFAR-10)", fontweight="bold")
    plt.xlabel("Throughput (img/sec, Higher is Better)")
    plt.ylabel("RBF MMD² (Lower is Better)")
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", frameon=True)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "plots" / "speed_vs_quality_pareto.png", dpi=200)
    plt.show()

# Display summary table preview
if all_dfs:
    display(master_df.head(20))


In [ ]:
# ==============================================================================
# CELL 10: PART 5 - AUTOMATED PACKAGING (.ZIP) & 1-CLICK KAGGLE DOWNLOAD
# ==============================================================================
print("=" * 80)
print("PART 5: PACKAGING BENCHMARK ARTIFACTS INTO ZIP ARCHIVE")
print("=" * 80)

import shutil
import zipfile
from IPython.display import FileLink, HTML

ZIP_PATH = Path("/kaggle/working/bd_comprehensive_benchmark_results_gentle.zip")

# 1. Clean previous archive if exists
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# 2. Package output directory using pure Python shutil (reliable across all environments)
print(f"[+] Creating zip archive of: {OUTPUT_DIR}")
shutil.make_archive(
    "/kaggle/working/bd_comprehensive_benchmark_results_gentle",
    "zip",
    root_dir=str(OUTPUT_DIR.parent),
    base_dir=str(OUTPUT_DIR.name),
)

# 3. Verify archive integrity and file size
if ZIP_PATH.exists():
    size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
    print(f"\n[✓] SUCCESS: Archive created successfully!")
    print(f"    Path: {ZIP_PATH}")
    print(f"    Size: {size_mb:.2f} MB")
    
    # 4. List packaged contents using pure Python zipfile
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        file_list = zf.namelist()
        print(f"\n[+] Packaged Artifacts Summary ({len(file_list)} files):")
        for name in file_list[:35]:
            print(f"    - {name}")

    print("\n" + "=" * 80)
    print("CLICK THE LINK BELOW TO DOWNLOAD BENCHMARK RESULTS:")
    print("=" * 80)
    display(FileLink("bd_comprehensive_benchmark_results_gentle.zip"))
else:
    print("[!] Error creating archive.")
